In [1]:
import os

# Set API keys from user-provided key
api_key = "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"
os.environ['OPENROUTER_API_KEY'] = api_key
os.environ['OPENAI_API_KEY'] = "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" # Also set for compatibility

print("API keys set.")

API keys set.


In [ ]:
!pip install requests pandas matplotlib seaborn plotly


In [ ]:
import requests
import json
import pandas as pd

# API endpoints
base_url = "http://localhost:3000"
endpoints = {
    "participants": "/api/admin/experimental-data?type=participants",
    "reaction_times": "/api/admin/experimental-data?type=reaction-times",
    "emotion_statistics": "/api/admin/emotion-analysis?action=get-statistics"
}

# Fetch data from all endpoints
api_data = {}
for key, endpoint in endpoints.items():
    try:
        response = requests.get(base_url + endpoint)
        response.raise_for_status()  # Raise an exception for bad status codes
        api_data[key] = response.json()
        print(f"Successfully fetched data from {key}")
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {key}: {e}")
        api_data[key] = None

# Display fetched data (optional)
# for key, data in api_data.items():
#     if data:
#         print(f"--- Data for {key} ---")
#         print(json.dumps(data, indent=2, ensure_ascii=False))


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Process the fetched data
if all(api_data.values()):
    participants_data = api_data['participants']
    sessions_data = api_data['reaction_times']
    emotions_data = api_data['emotion_statistics']

    # Helper functions to replicate JS logic
    def get_active_participants(participants):
        if 'data' in participants and participants['data']:
            return len([p for p in participants['data'] if p.get('hasVideoFiles')])
        return 0

    def get_completed_sessions(sessions):
        if 'data' in sessions and sessions['data']:
            return len([s for s in sessions['data'] if s.get('reactionTimeData') and len(s['reactionTimeData']) > 0])
        return 0
    
    def get_avg_reaction_time(sessions):
        if 'data' not in sessions or not sessions['data']:
            return 0
        
        total_reaction_time = 0
        num_sessions_with_reactions = 0
        
        for s in sessions['data']:
            if 'reactionTimeData' in s and s['reactionTimeData']:
                session_times = [r['reactionTimeMs'] for r in s['reactionTimeData'] if 'reactionTimeMs' in r]
                if session_times:
                    total_reaction_time += np.mean(session_times)
                    num_sessions_with_reactions += 1
        
        return total_reaction_time / num_sessions_with_reactions if num_sessions_with_reactions > 0 else 0


    # Replicate emotion trends and correlations generation
    def generate_emotion_trends(emotions):
        if not emotions:
            return []
        trends = []
        now = datetime.now()
        for i in range(6, -1, -1):
            date = now - timedelta(days=i)
            timestamp = date.strftime('%Y-%m-%d')
            for emotion in emotions:
                base_value = emotion.get('count', 0)
                variation = (np.random.random() - 0.5) * base_value * 0.3
                score = max(0, base_value + variation)
                trends.append({
                    'timestamp': timestamp,
                    'emotion': emotion.get('emotion'),
                    'score': round(score)
                })
        return trends

    def generate_emotion_correlations(emotions):
        if not emotions or len(emotions) < 2:
            return []
        correlations = []
        emotions_list = [e['emotion'] for e in emotions]
        for i in range(len(emotions_list)):
            for j in range(i + 1, len(emotions_list)):
                emotion1 = emotions_list[i]
                emotion2 = emotions_list[j]
                score1 = next((e['count'] for e in emotions if e['emotion'] == emotion1), 0)
                score2 = next((e['count'] for e in emotions if e['emotion'] == emotion2), 0)
                correlation = np.random.uniform(-1, 1)
                correlations.append({
                    'x': score1,
                    'y': score2,
                    'emotion': f"{emotion1}-{emotion2}",
                    'correlation': correlation
                })
        return correlations

    # Build the integrated analytics data dictionary
    total_analyses = emotions_data.get('data', {}).get('totalAnalyses', 1)
    if total_analyses == 0:
        total_analyses = 1 # Avoid division by zero
        
    dominant_emotions = emotions_data.get('data', {}).get('dominantEmotions', [])
    for e in dominant_emotions:
        e['percentage'] = (e.get('count', 0) / total_analyses) * 100

    integrated_data = {
        'storageStatus': {
            'blob': True,
            'kuzu': False,
            'filesystem': True
        },
        'participantStats': {
            'totalParticipants': participants_data.get('total', 0),
            'activeParticipants': get_active_participants(participants_data),
            'completionRate': participants_data.get('stats', {}).get('completionRate', 0),
            'averageSessionTime': 2700
        },
        'sessionStats': {
            'totalSessions': len(sessions_data.get('data', [])),
            'completedSessions': get_completed_sessions(sessions_data),
            'averageReactionTime': get_avg_reaction_time(sessions_data),
            'sessionDistribution': [
                {'name': '完了済み', 'value': get_completed_sessions(sessions_data)},
                {'name': '進行中', 'value': len(sessions_data.get('data', [])) - get_completed_sessions(sessions_data)}
            ]
        },
        'emotionStats': {
            'totalAnalyses': total_analyses if total_analyses != 1 else emotions_data.get('data', {}).get('totalAnalyses', 0),
            'dominantEmotions': dominant_emotions,
            'emotionTrends': generate_emotion_trends(dominant_emotions),
            'emotionCorrelations': generate_emotion_correlations(dominant_emotions)
        },
        'storageStats': {
            'blobUsage': participants_data.get('total', 0) * 1024,
            'kuzuUsage': 0,
            'totalDataPoints': (participants_data.get('total', 0) + 
                                len(sessions_data.get('data', [])) + 
                                emotions_data.get('data', {}).get('totalAnalyses', 0)),
            'syncStatus': 'synced'
        }
    }
    
    print("Data processed successfully.")
    # print(json.dumps(integrated_data, indent=2, ensure_ascii=False))

else:
    print("Could not process data due to fetching errors.")



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Note: You might need to install a Japanese font and rebuild the font cache.
# Example for macOS:
# 1. Download a font like Noto Sans JP.
# 2. Add it to your font directory (e.g., ~/Library/Fonts)
# 3. Find the font file path: `fm.findfont('Noto Sans JP')`
# If font is not found, specify the path to the font file.
try:
    # Try to find a common Japanese font.
    font_path = fm.findfont('Hiragino Sans', fallback_to_default=False)
    if not font_path:
        # Provide a fallback path if not found. This path might differ on your system.
        # On Colab or Debian-based systems: '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf' may work for English
        # For Japanese: '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
        # Check your system for available Japanese fonts.
        font_path = fm.findfont('IPAexGothic', fallback_to_default=True)
    
    plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
    plt.rcParams['axes.unicode_minus'] = False # handle minus sign correctly
    print(f"Using font: {plt.rcParams['font.family']}")
except:
    print("Defaulting to system font. Japanese characters may not display correctly.")


if 'integrated_data' in locals():
    # Participant Stats Visualization
    stats = integrated_data['participantStats']
    
    fig, axes = plt.subplots(2, 1, figsize=(10, 4), gridspec_kw={'height_ratios': [1, 3]})
    
    # Hide axes
    axes[0].axis('off')
    axes[1].axis('off')

    fig.suptitle('参加者統計 (Participant Statistics)', fontsize=16)

    # Key stats
    stat_labels = ['総参加者数', 'アクティブ', '完了率', '平均セッション時間']
    stat_values = [
        stats['totalParticipants'],
        stats['activeParticipants'],
        f"{stats['completionRate']:.1f}%",
        f"{stats['averageSessionTime'] / 60:.0f}分"
    ]
    colors = ['#3B82F6', '#10B981', '#8B5CF6', '#F97316']
    
    for i, (label, value, color) in enumerate(zip(stat_labels, stat_values, colors)):
        axes[0].text(i * 0.25 + 0.125, 0.5, f"{label}\n{value}", ha='center', va='center', fontsize=12, color=color, fontweight='bold')
        
    # Completion Rate Progress Bar
    completion_rate = stats['completionRate']
    axes[1].text(0, 0.6, '完了率 (Completion Rate)', ha='left', va='center', fontsize=12)
    axes[1].barh([0.3], [100], color='#E5E7EB', height=0.2)
    axes[1].barh([0.3], [completion_rate], color='#8B5CF6', height=0.2)
    axes[1].text(1.01, 0.3, f'{completion_rate:.1f}%', va='center', ha='left', fontsize=12)
    
    axes[1].set_xlim(0, 105)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

else:
    print("Run the data processing cell first.")


In [ ]:
if 'integrated_data' in locals():
    # Session Analysis Visualization
    stats = integrated_data['sessionStats']
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('セッション分析 (Session Analysis)', fontsize=16)

    # Session Completion Pie Chart
    session_dist = stats['sessionDistribution']
    labels = [d['name'] for d in session_dist]
    values = [d['value'] for d in session_dist]
    colors = ['#00C49F', '#FFBB28']
    
    axes[0].pie(values, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
    axes[0].set_title('セッション完了状況', fontsize=12)
    axes[0].axis('equal')

    # Reaction Time Stats
    axes[1].axis('off')
    axes[1].set_title('反応時間統計', fontsize=12, loc='center', y=0.8)

    rt_stats = {
        '平均反応時間': f"{stats['averageReactionTime']:.0f}ms",
        '総セッション数': stats['totalSessions'],
        '完了セッション': stats['completedSessions']
    }

    for i, (label, value) in enumerate(rt_stats.items()):
        axes[1].text(0.1, 0.6 - i * 0.2, f"{label}:", ha='left', va='center', fontsize=12)
        axes[1].text(0.9, 0.6 - i * 0.2, value, ha='right', va='center', fontsize=12, fontweight='bold')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

else:
    print("Run the data processing cell first.")


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set default theme for plotly
pio.templates.default = "plotly_white"

if 'integrated_data' in locals():
    # Emotion Analysis Visualization
    stats = integrated_data['emotionStats']
    dominant_emotions = pd.DataFrame(stats['dominantEmotions'])
    emotion_trends = pd.DataFrame(stats['emotionTrends'])
    emotion_correlations = pd.DataFrame(stats['emotionCorrelations'])
    
    colors = ['#8884d8', '#82ca9d', '#ffc658', '#ff7c7c', '#8dd1e1', '#d084d0']

    # Create subplots: 2 rows, 2 cols
    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{'type': 'pie'}, {'type': 'bar'}],
               [{'type': 'scatter', 'colspan': 2}, None]],
        subplot_titles=('感情分布 (Emotion Distribution)', 
                        '感情強度ランキング (Emotion Intensity)', 
                        '感情トレンド (Emotion Trends)'),
        vertical_spacing=0.15,
        horizontal_spacing=0.1
    )

    # Pie Chart for Emotion Distribution
    if not dominant_emotions.empty:
        fig.add_trace(go.Pie(
            labels=dominant_emotions['emotion'], 
            values=dominant_emotions['count'],
            hole=.3,
            marker_colors=colors
        ), row=1, col=1)

    # Bar Chart for Emotion Intensity
    if not dominant_emotions.empty:
        fig.add_trace(go.Bar(
            x=dominant_emotions['emotion'], 
            y=dominant_emotions['count'],
            marker_color=colors[0]
        ), row=1, col=2)
    
    # Line Chart for Emotion Trends
    if not emotion_trends.empty:
        # Pivot data for plotting
        trends_pivot = emotion_trends.pivot(index='timestamp', columns='emotion', values='score').reset_index()
        trends_pivot = trends_pivot.sort_values('timestamp')
        
        for i, emotion in enumerate(dominant_emotions['emotion'].head(3)):
             if emotion in trends_pivot.columns:
                fig.add_trace(go.Scatter(
                    x=trends_pivot['timestamp'], 
                    y=trends_pivot[emotion],
                    mode='lines+markers',
                    name=emotion,
                    line=dict(color=colors[i], width=2)
                ), row=2, col=1)

    # Update layout
    fig.update_layout(
        title_text='感情分析結果 (Emotion Analysis)',
        showlegend=True,
        height=700,
        width=900
    )
    
    # Update axes
    fig.update_xaxes(title_text="Emotion", row=1, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Score", row=2, col=1)
    
    fig.show()

else:
    print("Run the data processing cell first.")


In [ ]:
if 'integrated_data' in locals():
    # Storage and Sync Status
    storage_stats = integrated_data['storageStats']
    storage_status = integrated_data['storageStatus']
    
    fig = go.Figure()

    fig.add_trace(go.Indicator(
        mode = "gauge+number",
        value = storage_stats['blobUsage'] / 1024,
        domain = {'x': [0, 0.3], 'y': [0, 1]},
        title = {'text': "Blob Usage (KB)"},
        gauge = {'axis': {'range': [None, 1000]}, 'bar': {'color': "#3B82F6"}}
    ))
    
    fig.add_trace(go.Indicator(
        mode = "gauge+number",
        value = storage_stats['kuzuUsage'],
        domain = {'x': [0.35, 0.65], 'y': [0, 1]},
        title = {'text': "Kuzu Usage (KB)"},
        gauge = {'axis': {'range': [None, 1000]}, 'bar': {'color': "#10B981"}}
    ))

    fig.add_trace(go.Indicator(
        mode = "number",
        value = storage_stats['totalDataPoints'],
        domain = {'x': [0.7, 1], 'y': [0.5, 1]},
        title = {'text': "Total Data Points"}
    ))
    
    sync_status = storage_stats['syncStatus']
    status_text = {
        'synced': '✅ Synced',
        'syncing': '🔄 Syncing',
        'error': '❌ Error'
    }
    
    fig.add_annotation(
        x=0.85,
        y=0.2,
        text=f"Sync Status: <b>{status_text.get(sync_status, 'Unknown')}</b>",
        showarrow=False,
        font=dict(size=16)
    )

    fig.update_layout(
        title_text='ストレージ & 同期ステータス (Storage & Sync Status)',
        height=300
    )

    fig.show()

else:
    print("Run the data processing cell first.")
